In [1]:
from pynq import Overlay, allocate, PL
import numpy as np
import wave, struct, os, math, time

# =============================================================================
# 0) Load overlay and map DMAs
# =============================================================================
PL.reset()
overlay = Overlay("overlay5.bit")
overlay.download()

ip           = overlay.DeepWave_0
dma_inout    = overlay.axi_dma_inout      # MM2S -> in_stream (32-bit)
dma_param_bp = overlay.axi_dma_param_bp  # MM2S -> backproj param AXIS (32-bit)
dma_param_db = overlay.axi_dma_param_db  # MM2S -> deblur  param AXIS (32-bit)

regs = ip.register_map

print(regs)
print("✅ Overlay loaded.")

RegisterMap {
  goer_cfg_COS_OMEGA_0 = Register(goer_cfg_COS_OMEGA_0=write-only, RESERVED=write-only),
  goer_cfg_COS_OMEGA_1 = Register(goer_cfg_COS_OMEGA_1=write-only, RESERVED=write-only),
  goer_cfg_COS_OMEGA2_0 = Register(goer_cfg_COS_OMEGA2_0=write-only, RESERVED=write-only),
  goer_cfg_COS_OMEGA2_1 = Register(goer_cfg_COS_OMEGA2_1=write-only, RESERVED=write-only),
  goer_cfg_SIN_OMEGA_0 = Register(goer_cfg_SIN_OMEGA_0=write-only, RESERVED=write-only),
  goer_cfg_SIN_OMEGA_1 = Register(goer_cfg_SIN_OMEGA_1=write-only, RESERVED=write-only),
  debl_cfg = Register(debl_cfg=write-only, RESERVED=write-only),
  status_gz_samples_in = Register(status_gz_samples_in=0),
  status_gz_samples_in_ctrl = Register(status_gz_samples_in_ap_vld=0, RESERVED=0),
  status_gz_sample_win = Register(status_gz_sample_win=0),
  status_gz_sample_win_ctrl = Register(status_gz_sample_win_ap_vld=0, RESERVED=0),
  status_gz_samples_out = Register(status_gz_samples_out=0),
  status_gz_samples_out_ctrl = Registe

In [2]:
# =============================================================================
# 1) Load parameter CSVs
# =============================================================================
PARAM_DIR = "parameters"
WAVE_DIR  = "wave"

b_cols   = np.loadtxt(f"{PARAM_DIR}/b_vectors.csv", delimiter=",", skiprows=1, usecols=(2,3))
tau_f    = np.loadtxt(f"{PARAM_DIR}/tau.csv",        delimiter=",", skiprows=1)
lap_f    = np.loadtxt(f"{PARAM_DIR}/laplacian.csv",  delimiter=",", skiprows=1)
lap_offs = np.loadtxt(f"{PARAM_DIR}/lap_offsets.csv", delimiter=",", dtype=int)
theta    = np.loadtxt(f"{PARAM_DIR}/theta.csv", delimiter=",")

IMG_LEN = tau_f.shape[0]
ND      = lap_offs.shape[0]
N_ELEM  = (b_cols.shape[0]) // IMG_LEN
K       = theta.size - 1

print(f"[Params] IMG_LEN={IMG_LEN}, N_ELEM={N_ELEM}, ND={ND}, K={K}")

[Params] IMG_LEN=2234, N_ELEM=48, ND=6, K=22


In [3]:
# =============================================================================
# 4) Prepare WAV input
# =============================================================================
wav_file = os.path.join(WAVE_DIR, "two_speakers", "1-5.wav")
N_ELEM   = 48      # number of channels (from hardware)
N_WIN    = 200     # samples per block
GROUP_FRAMES = 9   # as in your C++ code

# -------------------------------------------------------------------------
# Load WAV
# -------------------------------------------------------------------------
wf = wave.open(wav_file, 'rb')
n_channels = wf.getnchannels()
n_samples   = wf.getnframes()
fs_in      = wf.getframerate()
samples = np.frombuffer(wf.readframes(n_samples), dtype='i2').reshape(-1, n_channels)
wf.close()

assert n_channels == N_ELEM, f"Invalid channel count ({n_channels} != {N_ELEM})"

# -------------------------------------------------------------------------
# Interleave like C++ testbench
# -------------------------------------------------------------------------
n_batches = n_samples // N_WIN
n_batches_group_aligned = (n_batches // GROUP_FRAMES) * GROUP_FRAMES
expected_frames = n_batches_group_aligned // GROUP_FRAMES

print(f"[WAV] {n_samples}×{n_channels}, fs={fs_in}")
print(f"[WAV] Using {n_batches_group_aligned} batches ({expected_frames} frames)")


def interleave_batches_apfixed(samples: np.ndarray, N_WIN: int, GROUP_FRAMES: int) -> np.ndarray:
    """
    Reorder 2D int16 audio data into 1D uint32 array matching ap_fixed<12,1> bit pattern.
    Each int16 sample is scaled and quantized as:
        sample_t = 16.0 * wav_sample / 32768.0
    and then converted to signed 12-bit two's-complement representation.
    """
    assert samples.dtype == np.int16, "Expected int16 input"
    n_samples, n_channels = samples.shape
    n_batches = n_samples // N_WIN

    # Truncate to integer number of GROUP_FRAMES batches
    n_batches = (n_batches // GROUP_FRAMES) * GROUP_FRAMES
    samples = samples[:n_batches * N_WIN, :]

    # Reshape to (n_batches, N_WIN, n_channels)
    samples_batched = samples.reshape(n_batches, N_WIN, n_channels)
    samples_batched = samples_batched.transpose(0, 2, 1)

    # Flatten
    flat_i16 = samples_batched.reshape(-1).astype(np.float64)

    # Apply same scaling as in C++
    scaled = 16.0 * flat_i16 / 32768.0

    # Quantize to 12-bit signed fixed-point (Q1.11)
    frac_bits = 11
    scaled_q = np.round(scaled * (1 << frac_bits)).astype(np.int32)

    # Clip to valid 12-bit signed range (-2048 .. +2047)
    scaled_q = np.clip(scaled_q, -2048, 2047)

    # Convert to two’s complement 12-bit representation (unsigned)
    scaled_u = np.where(scaled_q < 0, scaled_q + (1 << 12), scaled_q).astype(np.uint32)

    # Store in uint32 array (bitwise compatible with ap_fixed<12,1>)
    out_u32 = scaled_u

    return out_u32

samples_in_flattened = interleave_batches_apfixed(samples, N_WIN, GROUP_FRAMES)


[WAV] 56329×48, fs=16000
[WAV] Using 279 batches (31 frames)


In [4]:
# =============================================================================
# AXI-Lite config (matches current register map)
# =============================================================================

# We still need fs_in for Goertzel
# (we can get it from the WAV like we did)
FF    = 1666.67
fr    = float(fs_in) / float(N_WIN)
bin0  = int(round(FF / fr))
bins  = [bin0, bin0 - 1]

cos_omega  = [math.cos(2 * math.pi * b / N_WIN) for b in bins]
cos_omega2 = [2.0 * c for c in cos_omega]
sin_omega  = [math.sin(2 * math.pi * b / N_WIN) for b in bins]

def ap_fixed_to_u32(val, total_bits, int_bits, signed=True):
    frac = total_bits - int_bits
    scaled = int(round(float(val) * (1 << frac)))
    mask = (1 << total_bits) - 1
    if signed and scaled < 0:
        scaled = (scaled + (1 << total_bits)) & mask
    else:
        scaled &= mask
    return np.uint32(scaled)

# --- write Goertzel coefficients ---
regs.goer_cfg_COS_OMEGA_0  = ap_fixed_to_u32(cos_omega[0], 18, 2, True)
regs.goer_cfg_COS_OMEGA_1  = ap_fixed_to_u32(cos_omega[1], 18, 2, True)
regs.goer_cfg_COS_OMEGA2_0 = ap_fixed_to_u32(cos_omega2[0], 18, 2, True)
regs.goer_cfg_COS_OMEGA2_1 = ap_fixed_to_u32(cos_omega2[1], 18, 2, True)
regs.goer_cfg_SIN_OMEGA_0  = ap_fixed_to_u32(sin_omega[0], 18, 2, True)
regs.goer_cfg_SIN_OMEGA_1  = ap_fixed_to_u32(sin_omega[1], 18, 2, True)

# --- write deblur config ---
# your current HLS deblur_config only kept n_layers
# and you mapped it to ONE s_axilite register called `debl_cfg`
regs.debl_cfg = np.uint32(5)   # n_layers

print("[AXIL] Wrote Goertzel (6 regs) + deblur (1 reg).")

# -----------------------------------------------------------------------------
# Reset Goertzel status registers before first frame
# -----------------------------------------------------------------------------


def read_goertzel_status(regs):
    return {
        "samples_in": int(regs.status_gz_samples_in),
        "sample_win": int(regs.status_gz_sample_win),
        "samples_out": int(regs.status_gz_samples_out),
        "samples_out_fifo": int(regs.status_gz_samples_out_fifo),
    }

def read_crosscor_status(regs):
    """Read cross-correlation kernel status registers."""
    return {
        "state": int(regs.status_cc_state),
        "samples_in": int(regs.status_cc_samples_in),
        "samples_out": int(regs.status_cc_samples_out),
        "sample_idx": int(regs.status_cc_sample_idx),
        "norms_written": int(regs.status_cc_norms_written),
        "out_fifo": int(regs.status_cc_out_fifo),
        "norms_fifo": int(regs.status_cc_norms_fifo),
    }

def read_backproj_status(regs):
    """Read backprojection kernel status registers."""
    return {
        "config_loaded": bool(regs.status_bp_config_loaded),
        "fsm_state": int(regs.status_bp_fsm_state),
        "param_state": int(regs.status_bp_param_state),
        "idx": int(regs.status_bp_idx),
        "sigmas_in": int(regs.status_bp_sigmas_in),
        "pixels_out": int(regs.status_bp_pixels_out),
        "out_fifo_level": int(regs.status_bp_out_fifo_level),
    }

def read_deblur_status(regs):
    """Read deblur kernel status registers."""
    return {
        "config_loaded": bool(regs.status_db_config_loaded),
        "fsm_state": int(regs.status_db_fsm_state),
        "param_state": int(regs.status_db_param_state),
        "idx": int(regs.status_db_idx),
        "pixels_in": int(regs.status_db_pixels_in),
        "pixels_out": int(regs.status_db_pixels_out),
    }


print("[AXIL] Status counters:")
print("Goertzel : ", read_goertzel_status(regs))
print("CrossCor : ", read_crosscor_status(regs))
print("Backproj : ", read_backproj_status(regs))
print("Deblur   : ", read_deblur_status(regs))


[AXIL] Wrote Goertzel (6 regs) + deblur (1 reg).
[AXIL] Status counters:
Goertzel :  {'samples_in': 0, 'sample_win': 0, 'samples_out': 0, 'samples_out_fifo': 0}
CrossCor :  {'state': 0, 'samples_in': 0, 'samples_out': 0, 'sample_idx': 0, 'norms_written': 0, 'out_fifo': 0, 'norms_fifo': 0}
Backproj :  {'config_loaded': True, 'fsm_state': 0, 'param_state': 0, 'idx': 0, 'sigmas_in': 0, 'pixels_out': 0, 'out_fifo_level': 0}
Deblur   :  {'config_loaded': True, 'fsm_state': 0, 'param_state': 0, 'idx': 0, 'pixels_in': 0, 'pixels_out': 0}


In [5]:
# ============================================================
# 0) Vectorized quantizers (supporting negative wint)
# ============================================================
def q_to_u32_signed(x, ws: int, wint: int) -> np.ndarray:
    """Vectorized ap_fixed<ws,wint> → uint32 (two's complement)."""
    x = np.asarray(x, dtype=np.float64)
    frac_bits = ws - wint
    scale = 1 << frac_bits
    q = np.round(x * scale)
    q = np.clip(q, -(1 << (ws - 1)), (1 << (ws - 1)) - 1).astype(np.int64)
    return np.where(q < 0, q + (1 << ws), q).astype(np.uint32)


def q_to_u32_unsigned(x, ws: int, wint: int) -> np.ndarray:
    """Vectorized ap_ufixed<ws,wint> → uint32."""
    x = np.asarray(x, dtype=np.float64)
    frac_bits = ws - wint
    scale = 1 << frac_bits
    q = np.round(x * scale)
    q = np.clip(q, 0, (1 << ws) - 1).astype(np.int64)
    return q.astype(np.uint32)


# ============================================================
# 1) Fixed-point formats (from types.hpp)
# ============================================================
B_WS,  B_WINT  = 14, -2   # b_real_t  = ap_fixed<14, -2>
TAU_WS, TAU_WINT = 13, -3 # tau_t     = ap_fixed<13, -3>
LAP_WS, LAP_WINT = 15, -1 # lap_t     = ap_ufixed<15, -1>
TH_WS,  TH_WINT  = 18,  2 # theta_t   = ap_fixed<18,  2>


# ============================================================
# 2) Backprojection parameter buffer
# ============================================================
print("\n[Pack] Building backprojection parameter buffer...")

b_img_major = b_cols.reshape(IMG_LEN, N_ELEM, 2)
bre_all = b_img_major[:, :, 0].reshape(-1)
bim_all = b_img_major[:, :, 1].reshape(-1)

# quantize all re/im at once
bre_q = q_to_u32_signed(bre_all, B_WS, B_WINT)
bim_q = q_to_u32_signed(bim_all, B_WS, B_WINT)

# interleave re/im
bp_b = np.empty(bre_q.size * 2, dtype=np.uint32)
bp_b[0::2] = bre_q
bp_b[1::2] = bim_q

# quantize tau
tau_q = q_to_u32_signed(tau_f, TAU_WS, TAU_WINT)

# concatenate (b then tau)
bp_words = np.concatenate([bp_b, tau_q]).astype(np.uint32)
print(f"  → total {bp_words.size:,} words")

# allocate DMA buffer
param_bp_buf = allocate(shape=bp_words.shape, dtype=np.uint32)
param_bp_buf[:] = bp_words


# ============================================================
# 3) Deblur parameter buffer
# ============================================================
print("[Pack] Building deblur parameter buffer...")

lap_ddr = lap_f.reshape(-1)

# K (scalar)
K_u32 = np.array([np.uint32(K)], dtype=np.uint32)

# theta[0..K]
theta_q = q_to_u32_signed(theta[:K+1], TH_WS, TH_WINT)

# lap offsets
lap_offs_u32 = lap_offs.astype(np.uint32)

# lap_main
lap_main_q = q_to_u32_unsigned(lap_ddr[0], LAP_WS, LAP_WINT)

# lap_rest (ND × IMG_LEN)
lap_rest_f = lap_ddr[1 : 1 + ND * IMG_LEN].reshape(ND, IMG_LEN)
lap_rest_q = q_to_u32_unsigned(lap_rest_f, LAP_WS, LAP_WINT).reshape(-1)

# concatenate all pieces in C-order
db_words = np.concatenate([
    K_u32,
    theta_q.astype(np.uint32),
    lap_offs_u32,
    np.array([lap_main_q], dtype=np.uint32),
    lap_rest_q.astype(np.uint32)
])
print(f"  → total {db_words.size:,} words")

# allocate DMA buffer
param_db_buf = allocate(shape=db_words.shape, dtype=np.uint32)
param_db_buf[:] = db_words



[Pack] Building backprojection parameter buffer...
  → total 216,698 words
[Pack] Building deblur parameter buffer...
  → total 13,435 words


In [7]:
# ============================================================
# 4) Stream via DMA
# ============================================================
print("\n[Stream] Sending parameter sets...")

dma_param_bp.sendchannel.transfer(param_bp_buf)
dma_param_bp.sendchannel.wait()
print("  ✓ backprojection parameters streamed")

dma_param_db.sendchannel.transfer(param_db_buf)
dma_param_db.sendchannel.wait()
print("  ✓ deblur parameters streamed")

print("[Stream] All parameter DMA transfers complete.")



[Stream] Sending parameter sets...
  ✓ backprojection parameters streamed
  ✓ deblur parameters streamed
[Stream] All parameter DMA transfers complete.


In [8]:
def reconstruct_from_partial_sections(raw, bit_section=18, img_len=2234):
    """
    Reconstruct a complete frame from two partial, overlapping sections.

    Handles both even and odd frame lengths (n_pix = img_len + 1).
    """

    n_pix = img_len + 1

    # --- Extract section flag only (no bit shifting on data) --------------
    section = (raw >> bit_section) & 0x1

    # --- Detect transition 0→1 --------------------------------------------
    change_points = np.nonzero(np.diff(section) == 1)[0]
    if len(change_points) == 0:
        raise ValueError("No 0→1 transition found in section flag stream.")
    split_idx = change_points[0] + 1

    # --- Split at transition ----------------------------------------------
    first_part  = raw[:split_idx]     # section 0 (truncated at transition)
    second_part = raw[split_idx:]     # section 1 (starts after transition)

    # --- Determine split sizes (handle odd n_pix) -------------------------
    half0 = n_pix // 2 + (n_pix % 2)   # ceil(n_pix/2)
    half1 = n_pix // 2                 # floor(n_pix/2)

    # --- Combine head of section 1 + tail of section 0 --------------------
    head1 = second_part[:half1]
    tail0 = first_part[-half0:]

    reconstructed = np.concatenate([head1, tail0])
    return reconstructed

def check_indices(raw, img_len=2234):
    idx = raw >> 19
    exp = np.arange(img_len + 1, dtype=np.uint16)
    bad = np.nonzero(idx[:len(exp)] != exp)[0]
    if bad.size:
        print(f"[FAIL] {bad.size} bad indices, first at {bad[0]} (got {idx[bad[0]]}, expected {exp[bad[0]]})")
        return False
    return True

In [9]:
# Read all status registers
print("[AXIL] Status counters:")
print("Goertzel : ", read_goertzel_status(regs))
print("CrossCor : ", read_crosscor_status(regs))
print("Backproj : ", read_backproj_status(regs))
print("Deblur   : ", read_deblur_status(regs))


[AXIL] Status counters:
Goertzel :  {'samples_in': 0, 'sample_win': 0, 'samples_out': 0, 'samples_out_fifo': 0}
CrossCor :  {'state': 0, 'samples_in': 0, 'samples_out': 0, 'sample_idx': 0, 'norms_written': 0, 'out_fifo': 0, 'norms_fifo': 0}
Backproj :  {'config_loaded': True, 'fsm_state': 0, 'param_state': 0, 'idx': 2233, 'sigmas_in': 0, 'pixels_out': 0, 'out_fifo_level': 0}
Deblur   :  {'config_loaded': True, 'fsm_state': 0, 'param_state': 0, 'idx': 0, 'pixels_in': 0, 'pixels_out': 0}


In [10]:
# Stream all frames one by one
import time
samples_per_frame = GROUP_FRAMES * N_WIN * N_ELEM

in_buf_sel = allocate(shape=(samples_per_frame,), dtype=np.uint32)
out_buf = allocate(shape=(2*(IMG_LEN + 1),), dtype=np.uint32)

out_frames = np.zeros((expected_frames, IMG_LEN + 1), dtype=np.uint32)

# # --- Prime DMA pipeline ---
# dma_inout.recvchannel.transfer(out_buf)
# dma_inout.sendchannel.transfer(in_buf_sel)
# dma_inout.sendchannel.wait()
# dma_inout.recvchannel.wait()
# print("[Init] DMA primed")

print(f"[Run] Streaming samples to kernel, one frame at a time...")
for i in range(expected_frames):    
    in_buf_sel[:] = samples_in_flattened[i*samples_per_frame:(i+1)*samples_per_frame]
    
    dma_inout.recvchannel.transfer(out_buf)
    dma_inout.sendchannel.transfer(in_buf_sel)
    
    dma_inout.sendchannel.wait()
    dma_inout.recvchannel.wait()
    
    out_frames[i, :] = reconstruct_from_partial_sections(np.copy(out_buf),bit_section=18,img_len=IMG_LEN)
    idx_ok = check_indices(out_frames[i,:]) # Already check if all the indices are correct
    
    print(f" [Frame {i+1}/{expected_frames}] Done, reconstruction: {'OK' if idx_ok else 'FAILED'}")
print(f"Finished processing all samples and collecting the frames")

[Run] Streaming samples to kernel, one frame at a time...
 [Frame 1/31] Done, reconstruction: OK
 [Frame 2/31] Done, reconstruction: OK
 [Frame 3/31] Done, reconstruction: OK
 [Frame 4/31] Done, reconstruction: OK
 [Frame 5/31] Done, reconstruction: OK
 [Frame 6/31] Done, reconstruction: OK
 [Frame 7/31] Done, reconstruction: OK
 [Frame 8/31] Done, reconstruction: OK
 [Frame 9/31] Done, reconstruction: OK
 [Frame 10/31] Done, reconstruction: OK
 [Frame 11/31] Done, reconstruction: OK
 [Frame 12/31] Done, reconstruction: OK
 [Frame 13/31] Done, reconstruction: OK
 [Frame 14/31] Done, reconstruction: OK
 [Frame 15/31] Done, reconstruction: OK
 [Frame 16/31] Done, reconstruction: OK
 [Frame 17/31] Done, reconstruction: OK
 [Frame 18/31] Done, reconstruction: OK
 [Frame 19/31] Done, reconstruction: OK
 [Frame 20/31] Done, reconstruction: OK
 [Frame 21/31] Done, reconstruction: OK
 [Frame 22/31] Done, reconstruction: OK
 [Frame 23/31] Done, reconstruction: OK
 [Frame 24/31] Done, reconstruc

In [16]:
# For some reason values only get written after 147 samples, skipping the first 147.

norm_bits = out_frames[:,0] & ((1<<18)-1)
norm = norm_bits / float(1<<11)

pixel_bits = out_frames[:,1:] & ((1<<18)-1)
pixels = pixel_bits / float(1<<16) / (2048*np.tanh(1))

In [14]:
# --- Precompute indices ---
frames, pixels_per_frame = pixels.shape
frame_idx = np.repeat(np.arange(frames), pixels_per_frame)
pixel_idx = np.tile(np.arange(pixels_per_frame), frames)
values    = pixels.ravel()

# --- Write pixel CSV (frame,pixel,value) ---
np.savetxt(
    "deepwave_sim_hw.csv",
    np.column_stack((frame_idx, pixel_idx, values)),
    delimiter=",",
    header="frame,pixel,value",
    comments="",
    fmt=["%d", "%d", "%.8e"]  # 👈 integers for frame/pixel, float for value
)

# --- Write norms CSV (frame,norm) ---
np.savetxt(
    "deepwave_norms_hw.csv",
    np.column_stack((np.arange(norm.size), norm)),
    delimiter=",",
    header="frame,norm",
    comments="",
    fmt=["%d", "%.8e"]
)

print("[CSV] Wrote deepwave_sim_hw.csv and deepwave_norms_hw.csv")

[CSV] Wrote deepwave_sim_hw.csv and deepwave_norms_hw.csv
